In [1]:
import os
import urllib.request
from pathlib import Path

import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from xarray.coding.times import decode_cf_datetime

# Two experiments that differ only in the DATM atmospheric forcing.
EXPS = {
    "JRA55": "hurcn_gomcar_ocn_wav.001",
    "ERA5":  "hurcn_gomcar_ocn_wav_era5.001",
}
# CESM's default short-term archive on Derecho (DOUT_S_ROOT). $SCRATCH points at your Derecho
# scratch from both Derecho and Casper. If you changed DOUT_S_ROOT, point ARCHIVE there instead.
ARCHIVE = Path(os.environ["SCRATCH"]) / "archive"

# Hurricane Laura from IBTrACS v04r01, North Atlantic subset (NOAA NCEI, doi:10.25921/82ty-9e16):
# https://www.ncei.noaa.gov/products/international-best-track-archive
# The file is downloaded on first use and reused afterwards.
IBTRACS_URL = ("https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/"
               "v04r01/access/csv/ibtracs.NA.list.v04r01.csv")
IBTRACS_CSV = Path("/glade/u/home/shanru/work/crocodile2026/workspace/data/ibtracs/ibtracs.NA.list.v04r01.csv")


def load_ibtracs_storm(name, season, csv=IBTRACS_CSV):
    """One storm's IBTrACS track, using the US agency (NHC HURDAT2) wind, pressure and record type."""
    if not csv.exists():
        csv.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(IBTRACS_URL, csv)
    cols = ["SID", "SEASON", "NAME", "ISO_TIME", "LAT", "LON", "IFLAG",
            "USA_WIND", "USA_PRES", "USA_RECORD", "USA_STATUS"]
    # Row 2 of IBTrACS CSVs holds units; missing values are single blanks.
    df = pd.read_csv(csv, skiprows=[1], usecols=cols, na_values=[" "], keep_default_na=False, low_memory=False)
    df = df[(df["NAME"] == name) & (df["SEASON"].astype(int) == season)]
    if df["SID"].nunique() != 1:
        raise ValueError(f"Expected one storm named {name} in {season}, found {df['SID'].unique()}")
    df = df.dropna(subset=["USA_WIND"])
    return {
        "sid":    df["SID"].iloc[0],
        "time":   pd.to_datetime(df["ISO_TIME"]).values.astype("datetime64[m]"),
        "lat":    df["LAT"].astype(float).values,
        "lon":    df["LON"].astype(float).values,
        "wind":   df["USA_WIND"].astype(float).values,
        "pres":   df["USA_PRES"].astype(float).values,
        "status": df["USA_STATUS"].values,
        "record": df["USA_RECORD"].fillna("").values,
        # IFLAG "O" = original agency fix; "P"/"I" = IBTrACS 3-hourly interpolation
        "observed": df["IFLAG"].str[0].eq("O").values,
    }


TRACK = load_ibtracs_storm("LAURA", 2020)
LANDFALLS = np.flatnonzero(TRACK["record"] == "L")
# Strongest landfall, 2020-08-27 06 UTC near Cameron, Louisiana
i_lf = LANDFALLS[np.argmax(TRACK["wind"][LANDFALLS])]
LANDFALL = (TRACK["lon"][i_lf], TRACK["lat"][i_lf])

# Saffir-Simpson categories by maximum sustained wind (kt)
CATEGORIES = [("TD", 0, "#5ebaff"), ("TS", 34, "#00faf4"), ("Cat 1", 64, "#ffffcc"),
              ("Cat 2", 83, "#ffe775"), ("Cat 3", 96, "#ffc140"), ("Cat 4", 113, "#ff8f20"),
              ("Cat 5", 137, "#ff6060")]


def category_color(wind_kt):
    return [color for _, lo, color in CATEGORIES if wind_kt >= lo][-1]


def track_position(when):
    """Best-track position linearly interpolated to a time, or None outside the track."""
    t = np.datetime64(when, "m").astype(float)
    tt = TRACK["time"].astype("datetime64[m]").astype(float)
    if t < tt[0] or t > tt[-1]:
        return None
    return np.interp(t, tt, TRACK["lon"]), np.interp(t, tt, TRACK["lat"])


def hist_dir(case):
    return ARCHIVE / case / "ocn" / "hist"


def open_sst(case):
    """Daily-mean SST (tos) from the MOM6 sfc history files of one case."""
    files = sorted(hist_dir(case).glob(f"{case}.mom6.h.sfc.*.nc"))
    ds = xr.open_mfdataset(files, combine="by_coords", decode_times=False)
    # MOM6 writes "days since 0001-01-01" with calendar="gregorian", but its clock is proleptic
    # Gregorian. Decoding with the mixed Julian/Gregorian calendar puts every date 2 days early.
    time = decode_cf_datetime(ds["time"].values, ds["time"].attrs["units"], calendar="proleptic_gregorian")
    return ds["tos"].assign_coords(time=time)


static = xr.open_dataset(hist_dir(EXPS["JRA55"]) / f"{EXPS['JRA55']}.mom6.h.static.nc")
WET  = xr.DataArray(static["wet"].values > 0, dims=("yh", "xh"))
AREA = xr.DataArray(static["areacello"].values, dims=("yh", "xh")).where(WET)
# Cell corners for pcolormesh, shifted from 0-360 to -180-180 so cartopy needs no wrapping.
LONC = (static["geolon_c"].values + 180.0) % 360.0 - 180.0
LATC = static["geolat_c"].values
LON  = (static["geolon"].values + 180.0) % 360.0 - 180.0
LAT  = static["geolat"].values

sst = {name: open_sst(case).where(WET).load() for name, case in EXPS.items()}
if not np.array_equal(sst["JRA55"].time.values, sst["ERA5"].time.values):
    raise ValueError("The two experiments have different time axes")

TIMES = sst["JRA55"].time.values
DATES = np.datetime_as_string(TIMES, unit="D")
NDAYS = len(TIMES)

PROJ   = ccrs.PlateCarree()
EXTENT = [LONC.min(), LONC.max(), LATC.min(), LATC.max()]


def add_track(ax, legend=False, now=None):
    """Overlay Laura's IBTrACS track: line through all 3-hourly points, NHC fixes coloured by
    category, 00 UTC dates labelled, landfalls marked. If `now` is given, circle the storm there."""
    obs = TRACK["observed"]
    ax.plot(TRACK["lon"], TRACK["lat"], color="k", linewidth=1.2, transform=PROJ, zorder=5)
    ax.scatter(TRACK["lon"][obs], TRACK["lat"][obs], c=[category_color(w) for w in TRACK["wind"][obs]],
               s=22, edgecolors="k", linewidths=0.5, transform=PROJ, zorder=6)
    for t, x, y in zip(TRACK["time"], TRACK["lon"], TRACK["lat"]):
        if str(t).endswith("T00:00"):
            ax.text(x + 0.6, y - 0.9, str(t)[8:10], fontsize=7, transform=PROJ, zorder=6)
    ax.scatter(TRACK["lon"][LANDFALLS], TRACK["lat"][LANDFALLS], marker="v", s=28, color="k",
               transform=PROJ, zorder=7)
    ax.plot(*LANDFALL, marker="*", color="k", markersize=11, transform=PROJ, zorder=7)
    if now is not None and (pos := track_position(now)) is not None:
        ax.plot(*pos, marker="o", markersize=12, markerfacecolor="none", markeredgecolor="m",
                markeredgewidth=2, transform=PROJ, zorder=7)
    if legend:
        handles = [plt.Line2D([], [], marker="o", linestyle="", markerfacecolor=c, markeredgecolor="k",
                              label=name) for name, _, c in CATEGORIES if name != "Cat 5"]
        handles.append(plt.Line2D([], [], marker="v", linestyle="", color="k", label="landfall"))
        ax.legend(handles=handles, loc="lower left", fontsize=7, title="Laura (IBTrACS)", title_fontsize=8,
                  framealpha=0.9).set_zorder(8)


def add_map(ax, labels=True, legend=False, now=None):
    ax.set_extent(EXTENT, crs=PROJ)
    ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=2)
    ax.coastlines(resolution="50m", linewidth=0.6, zorder=3)
    gl = ax.gridlines(draw_labels=labels, linewidth=0.3, color="0.5", alpha=0.5)
    gl.top_labels = gl.right_labels = False
    add_track(ax, legend=legend, now=now)


def area_mean(da):
    return (da * AREA).sum(("yh", "xh")) / AREA.where(da.notnull()).sum(("yh", "xh"))


print(f"Laura {TRACK['sid']}: {len(TRACK['time'])} track points {TRACK['time'][0]} to {TRACK['time'][-1]}, "
      f"peak {TRACK['wind'].max():.0f} kt / {np.nanmin(TRACK['pres']):.0f} hPa, "
      f"{len(LANDFALLS)} landfalls, strongest at {LANDFALL[1]:.1f}N {-LANDFALL[0]:.1f}W")
print(f"{NDAYS} daily means: {DATES[0]} to {DATES[-1]}")
for name, da in sst.items():
    print(f"{name:6s} {EXPS[name]}  SST range {float(da.min()):.2f} to {float(da.max()):.2f} degC")

OSError: no files to open

In [ ]:
# Run-mean SST in each experiment, and ERA5 minus JRA55
mean = {name: da.mean("time") for name, da in sst.items()}
diff = mean["ERA5"] - mean["JRA55"]

vmin, vmax = np.nanpercentile(np.stack([m.values for m in mean.values()]), [1, 99])
dlim = float(np.nanpercentile(np.abs(diff.values), 99))

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5), subplot_kw={"projection": PROJ}, constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes[:2], EXPS)):
    pc = ax.pcolormesh(LONC, LATC, mean[name], transform=PROJ, cmap="RdYlBu_r", vmin=vmin, vmax=vmax)
    add_map(ax, legend=(i == 0))
    ax.set_title(f"{name}: mean SST")
fig.colorbar(pc, ax=axes[:2], shrink=0.85, label="SST (\u00b0C)")

pd_ = axes[2].pcolormesh(LONC, LATC, diff, transform=PROJ, cmap="RdBu_r", vmin=-dlim, vmax=dlim)
add_map(axes[2])
axes[2].set_title("ERA5 \u2212 JRA55")
fig.colorbar(pd_, ax=axes[2], shrink=0.85, label="\u0394SST (\u00b0C)")
fig.suptitle(f"Mean SST, {DATES[0]} to {DATES[-1]}  (Laura best track, dates at 00 UTC; \u2605 landfall)", fontsize=14)
plt.show()

print(f"ERA5 - JRA55: area mean {float(area_mean(diff)):+.3f} degC, "
      f"RMS {float(np.sqrt(area_mean(diff**2))):.3f} degC, "
      f"min {float(diff.min()):+.2f}, max {float(diff.max()):+.2f}")

In [ ]:
# Daily SST snapshots around Laura's passage: JRA55, ERA5, and ERA5 minus JRA55
SNAP_DATES = ["2020-08-24", "2020-08-26", "2020-08-28", "2020-08-30"]

snaps = {name: da.sel(time=SNAP_DATES, method="nearest") for name, da in sst.items()}
snap_diff = snaps["ERA5"] - snaps["JRA55"]
slim = float(np.nanpercentile(np.abs(snap_diff.values), 99))

rows = [("JRA55", snaps["JRA55"], "RdYlBu_r", vmin, vmax),
        ("ERA5",  snaps["ERA5"],  "RdYlBu_r", vmin, vmax),
        ("ERA5 \u2212 JRA55", snap_diff, "RdBu_r", -slim, slim)]

fig, axes = plt.subplots(3, len(SNAP_DATES), figsize=(5 * len(SNAP_DATES), 11),
                         subplot_kw={"projection": PROJ}, constrained_layout=True)
for r, (label, data, cmap, lo, hi) in enumerate(rows):
    for c in range(len(SNAP_DATES)):
        ax = axes[r, c]
        pc = ax.pcolormesh(LONC, LATC, data.isel(time=c), transform=PROJ, cmap=cmap, vmin=lo, vmax=hi)
        add_map(ax, labels=(c == 0 or r == 2), legend=(r == 0 and c == 0),
                now=data.time.values[c])
        day = np.datetime_as_string(data.time.values[c], unit="D")
        ax.set_title(f"{label}  {day}", fontsize=10)
    fig.colorbar(pc, ax=axes[r, :], shrink=0.9, pad=0.01,
                 label="\u0394SST (\u00b0C)" if r == 2 else "SST (\u00b0C)")
fig.suptitle("Daily-mean SST  (magenta circle: Laura's position at 12 UTC that day)", fontsize=14)
plt.show()

In [ ]:
# SST change over the run (last day minus first day): the hurricane cold wake in each experiment
change = {name: da.isel(time=-1) - da.isel(time=0) for name, da in sst.items()}
change_diff = change["ERA5"] - change["JRA55"]
clim = float(np.nanpercentile(np.abs(np.stack([c.values for c in change.values()])), 99.5))
cdlim = float(np.nanpercentile(np.abs(change_diff.values), 99))

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5), subplot_kw={"projection": PROJ}, constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes[:2], EXPS)):
    pc = ax.pcolormesh(LONC, LATC, change[name], transform=PROJ, cmap="RdBu_r", vmin=-clim, vmax=clim)
    add_map(ax, legend=(i == 0))
    ax.set_title(f"{name}: SST {DATES[-1]} \u2212 {DATES[0]}")
fig.colorbar(pc, ax=axes[:2], shrink=0.85, label="\u0394SST (\u00b0C)")

pd_ = axes[2].pcolormesh(LONC, LATC, change_diff, transform=PROJ, cmap="PuOr_r", vmin=-cdlim, vmax=cdlim)
add_map(axes[2])
axes[2].set_title("ERA5 change \u2212 JRA55 change")
fig.colorbar(pd_, ax=axes[2], shrink=0.85, label="\u0394SST (\u00b0C)")
fig.suptitle("SST change over the run  (Laura best track, dates at 00 UTC; \u2605 landfall)", fontsize=14)
plt.show()

for name, c in change.items():
    print(f"{name:6s} max cooling {float(c.min()):+.2f} degC, area-mean change {float(area_mean(c)):+.3f} degC")

In [ ]:
# Daily area-mean SST in each experiment, and the area-mean and RMS of ERA5 minus JRA55
daily_diff = sst["ERA5"] - sst["JRA55"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5), constrained_layout=True)
for name, da in sst.items():
    ax1.plot(TIMES, area_mean(da), marker="o", label=name)
ax1.set_ylabel("area-mean SST (\u00b0C)")
ax1.set_title("Domain-mean SST")
ax1.legend()

ax2.plot(TIMES, area_mean(daily_diff), marker="o", label="mean")
ax2.plot(TIMES, np.sqrt(area_mean(daily_diff**2)), marker="s", label="RMS")
ax2.axhline(0, color="k", linewidth=0.6)
ax2.set_ylabel("\u0394SST (\u00b0C)")
ax2.set_title("ERA5 \u2212 JRA55")
ax2.legend()

for ax in (ax1, ax2):
    ax.axvline(np.datetime64("2020-08-27T06"), color="0.5", linestyle="--", linewidth=0.8)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)
plt.show()